# ICL-PILOT NeMo Data Designer Colab

This notebook clones the repo, installs the project plus `data-designer`, builds the existing 4-year-old custom generation package, and then runs a NeMo Data Designer version of the two-stage story generation workflow.

Design choice: this notebook does **not** replace the repo's custom schedule logic. It uses the repo's frozen roster and story schedule as the seed dataset for Data Designer so the comparison stays aligned with the existing experiment.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/shamira-venturini/ICL-PILOT.git"
REPO_DIR = Path("/content/ICL-PILOT")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

%cd /content/ICL-PILOT

In [ ]:
%pip -q install -U pip setuptools wheel
%pip -q install -e . data-designer pydantic

In [ ]:
import sys
from pathlib import Path

SRC_DIR = Path('/content/ICL-PILOT/src')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import icl_pilot
print('icl_pilot import OK from', icl_pilot.__file__)

## Model And Run Settings

The project report names `Meta-Llama-3.2-8B-Instruct` as the generation model. Data Designer's welcome docs say it can work with endpoints such as NVIDIA, OpenAI, and vLLM, but the default Colab install only preconfigures some providers. So this notebook now makes the provider check explicit before generation. `N_REPLICATES=2` is a cheap smoke-test; switch to `10` when you want the full paper-style package.

In [ ]:
# Inspect the providers and model aliases currently registered in this Data Designer install.
!data-designer config list

In [ ]:
import os
from getpass import getpass

# The report uses Meta-Llama-3.2-8B-Instruct.
# Choose a provider that actually appears in `data-designer config list` above.
# The Data Designer docs list NVIDIA/OpenAI/OpenRouter as default providers and point to CLI
# configuration for custom providers/models. If your open-weights vLLM endpoint is not listed,
# it must be registered first before this notebook can use it.
# Uncomment one of these only if your chosen provider actually requires it.
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
# os.environ["NVIDIA_API_KEY"] = getpass("NVIDIA_API_KEY: ")

MODEL_PROVIDER = "openai"   # replace with a provider name that appears in `data-designer config list`
MODEL_NAME = "Meta-Llama-3.2-8B-Instruct"
MODEL_ALIAS = "story-model"

N_REPLICATES = 2
PREVIEW_RECORDS = 3
RUN_FULL_CREATE = False

print({
    "MODEL_PROVIDER": MODEL_PROVIDER,
    "MODEL_NAME": MODEL_NAME,
    "N_REPLICATES": N_REPLICATES,
    "PREVIEW_RECORDS": PREVIEW_RECORDS,
    "RUN_FULL_CREATE": RUN_FULL_CREATE,
})

In [ ]:
from icl_pilot.generation import build_generation_package

artifacts = build_generation_package(
    dev_measures_csv="phase2/measures/dev_measures.csv",
    severity_profile_csv="phase2/measures/severity_profile_banded_table.csv",
    counterbalance_csv="configs/generation/cross_story_counterbalancing.csv",
    output_dir="configs/generation/colab_generation_package_nemo_dd",
    n_replicates=N_REPLICATES,
)

artifacts

In [ ]:
import json
import pandas as pd

schedule_df = pd.read_csv(artifacts.schedule_csv)
narratives_df = pd.read_csv("phase2/story_units/story_unit_narratives.csv")
story_reference = json.loads(Path("phase2/story_units/story_unit_reference.json").read_text())

def numbered_lines(lines):
    return "\n".join(f"{i + 1}. {line}" for i, line in enumerate(lines))

def bullet_lines(lines):
    if not lines:
        return "None"
    return "\n".join(f"- {line}" for line in lines)

def exemplar_lookup(frame):
    table = frame[["File_ID", "Group", "story_id", "story_text"]].copy()
    table["File_ID"] = table["File_ID"].astype(str)
    table["Group"] = table["Group"].astype(str)
    table["story_id"] = table["story_id"].astype(str)
    return {
        (row.File_ID, row.Group, row.story_id): row.story_text
        for row in table.itertuples(index=False)
    }

story_lookup = exemplar_lookup(narratives_df)

seed_df = schedule_df.copy()
seed_df["story_title"] = seed_df["target_story"].map(lambda story_id: story_reference[story_id]["title"])
seed_df["story_units_text"] = seed_df["target_story"].map(
    lambda story_id: numbered_lines(story_reference[story_id]["units"])
)
seed_df["content_invariants_text"] = seed_df["target_story"].map(
    lambda story_id: bullet_lines(story_reference[story_id].get("content_invariants", []))
)
seed_df["optional_content_text"] = seed_df["target_story"].map(
    lambda story_id: bullet_lines(story_reference[story_id].get("optional_content", []))
)
seed_df["prompt_constraints_text"] = seed_df["target_story"].map(
    lambda story_id: bullet_lines(story_reference[story_id].get("prompt_constraints", []))
)
seed_df["prompt_td_exemplar_story"] = seed_df.apply(
    lambda row: story_lookup.get((str(int(row["td_child_id"])), "TD", str(row["prompt_td_story_e1"])), ""),
    axis=1,
)
seed_df["prompt_sli_exemplar_story"] = seed_df.apply(
    lambda row: story_lookup.get((str(int(row["sli_child_id"])), "SLI", str(row["prompt_sli_story_e2"])), ""),
    axis=1,
)

seed_csv = Path("data/processed/nemo_data_designer/four_year_old_generation_seed.csv")
seed_csv.parent.mkdir(parents=True, exist_ok=True)
seed_df.to_csv(seed_csv, index=False)

print(f"Seed rows: {len(seed_df)}")
seed_df[[
    "bundle_id",
    "target_story",
    "story_title",
    "sli_child_id",
    "td_child_id",
    "prompt_td_story_e1",
    "prompt_sli_story_e2",
]].head()

## Configure NeMo Data Designer

This version uses the repo's schedule rows as seed data and asks Data Designer to produce:

1. a structured stage-1 TD-like content plan
2. a stage-1 TD-like child story
3. a stage-2 SLI-like transformation that preserves the same ENNI events

In [ ]:
import data_designer.config as dd
from data_designer.interface import DataDesigner
from pydantic import BaseModel, Field


class StoryPlan(BaseModel):
    clauses: list[str] = Field(
        description="Four to eight short child-like clauses that preserve the story spine order."
    )
    omitted_optional_details: list[str] = Field(
        default_factory=list,
        description="Optional details intentionally left out of the plan."
    )


model_configs = [
    dd.ModelConfig(
        alias=MODEL_ALIAS,
        model=MODEL_NAME,
        provider=MODEL_PROVIDER,
    )
]

config_builder = dd.DataDesignerConfigBuilder(model_configs=model_configs)
config_builder.with_seed_dataset(dd.LocalFileSeedSource(path=str(seed_csv)))

config_builder.add_column(
    dd.LLMStructuredColumnConfig(
        name="stage1_td_plan",
        model_alias=MODEL_ALIAS,
        prompt="""You are writing a content plan for a typically developing child's ENNI retell.

Target story: {{ target_story }} ({{ story_title }})
Canonical story spine:
{{ story_units_text }}

Content that should stay stable:
{{ content_invariants_text }}

Optional details:
{{ optional_content_text }}

Constraints:
{{ prompt_constraints_text }}

Example natural TD story from another ENNI story slot:
{{ prompt_td_story_e1 }}: {{ prompt_td_exemplar_story }}

Write 4 to 8 short clauses in order.
Keep the plan child-like and simple.
Do not add new major events.
Use the TD example mainly for narrative style and developmental level, not for its content.
""",
        output_format=StoryPlan,
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="stage1_td_story",
        model_alias=MODEL_ALIAS,
        prompt="""You are generating a short TD-like child retell of an ENNI story.

Target story: {{ target_story }} ({{ story_title }})
Story plan clauses: {{ stage1_td_plan.clauses }}
Prompt TD exemplar story slot: {{ prompt_td_story_e1 }}
Example natural TD story: {{ prompt_td_exemplar_story }}

Keep the event order.
Do not add new major events.
Keep the language simple, concrete, and child-like.
Respond with only the story text.
""",
    )
)

config_builder.add_column(
    dd.LLMTextColumnConfig(
        name="stage2_sli_story",
        model_alias=MODEL_ALIAS,
        prompt="""You are transforming a TD-like child story so it better matches the target child's language profile while preserving the same ENNI events.

Target story: {{ target_story }} ({{ story_title }})
Target child age: {{ sli_age }}
Target severity band: {{ sli_severity_band }}
Target profile label: {{ sli_profile_label }}
Prompt SLI exemplar story slot: {{ prompt_sli_story_e2 }}
Example natural SLI story: {{ prompt_sli_exemplar_story }}

Original TD-like story:
{{ stage1_td_story }}

Canonical story spine:
{{ story_units_text }}

Preserve the story content and order.
Allow shorter clauses, clinically plausible simplification, and omission of optional details.
Use the SLI exemplar mainly as a style/profile guide, not as content to copy.
Do not invent new major events.
Do not turn the output into an adult paraphrase.
Respond with only the transformed story text.
""",
    )
)

data_designer = DataDesigner()
data_designer.validate(config_builder)
config_builder

In [ ]:
preview = data_designer.preview(config_builder, num_records=min(PREVIEW_RECORDS, len(seed_df)))
preview.display_sample_record()
preview.dataset[[
    "bundle_id",
    "target_story",
    "stage1_td_story",
    "stage2_sli_story",
]].head(PREVIEW_RECORDS)

In [ ]:
if RUN_FULL_CREATE:
    results = data_designer.create(
        config_builder,
        num_records=len(seed_df),
        dataset_name="icl-pilot-nemo-data-designer",
    )
    generated_df = results.dataset
    out_csv = Path("data/generated/nemo_data_designer/four_year_old_story_generations.csv")
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    generated_df.to_csv(out_csv, index=False)
    print(f"Saved {len(generated_df)} rows to {out_csv}")
    display(generated_df[[
        "bundle_id",
        "target_story",
        "stage1_td_story",
        "stage2_sli_story",
    ]].head())
else:
    print("RUN_FULL_CREATE is False. Inspect the preview first, then switch RUN_FULL_CREATE = True.")